# MvDEC S2 last arms: L2-balanced Fig. 2 concat and DEKM paper-text input normalisation

Controlled diagnostic only (`controlled_diagnostic_not_exact_paper_truth`). Two one-field
changes to S2 (`mvdec_2025_scratch_both_views_true_bottleneck_min500_release_l2_v1`):

- `..._figure2_l2_concat_min500_release_l2_v1`: refinement and evaluation use
  `concat(l2_normalize(h_view1), l2_normalize(h_view2))`, so View1 no longer dominates
  the concatenation by scale (the raw-concat arm returned fused == View1 in every seed).
- `..._min500_paper_rms_v1`: inputs scaled so `(1/d)||x||^2 = 1` (DEKM paper text, H9)
  instead of the release L2 unit norm.

REUTERS seeds 42-51 for both, plus 20NEWS seeds 42-44 for the L2 concat arm. Each run's
stdout/stderr is kept in `logs/` so a failure can be diagnosed. A label trajectory
sidecar is written per run; it never feeds back.

Kaggle settings: GPU on, Internet on, no input dataset.

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__)
print("GPUs:", gpus)
assert gpus, "Enable a Kaggle GPU before running this diagnostic."

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys

PINNED_COMMIT = "e84c305d6dd3b0260a77b6137954a517349f3d85"
BRANCH = "claude/mvdec-hypotheses-2026-10-24"
REPO_URL = "https://github.com/vdat-18/mvdec_fpmax.git"
REPO = Path("/kaggle/temp/mvdec_fpmax")
OUTPUT = Path("/kaggle/temp/mvdec-s2-last-arms")
RESULTS = Path("/kaggle/working/mvdec-s2-last-arms")
L2_CONCAT = "mvdec_2025_scratch_both_views_true_bottleneck_figure2_l2_concat_min500_release_l2_v1"
PAPER_RMS = "mvdec_2025_scratch_both_views_true_bottleneck_min500_paper_rms_v1"
PROTOCOLS = {L2_CONCAT, PAPER_RMS}
RUNS = (
    [(L2_CONCAT, "REUTERS", s) for s in range(42, 52)]
    + [(PAPER_RMS, "REUTERS", s) for s in range(42, 52)]
    + [(L2_CONCAT, "20NEWS", s) for s in range(42, 45)]
)

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


for path in (REPO, OUTPUT, RESULTS):
    if path.exists():
        shutil.rmtree(path)
OUTPUT.mkdir(parents=True)
(RESULTS / "trajectory").mkdir(parents=True)
(RESULTS / "logs").mkdir()
subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "checkout", "-q", PINNED_COMMIT], cwd=REPO, check=True)
TRAINING_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert TRAINING_COMMIT == PINNED_COMMIT, (TRAINING_COMMIT, PINNED_COMMIT)
assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip()
print("Training commit:", TRAINING_COMMIT)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "gdown"], check=True)
sys.path.insert(0, str(REPO / "src"))
subprocess.run(
    [sys.executable, str(REPO / "scripts/fetch_dekm_release.py"), "--datasets", *sorted({d for _, d, _ in RUNS})],
    cwd=REPO,
    check=True,
)

In [ ]:
subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q",
        "tests/test_mvdec_independent_view2.py",
        "tests/test_mvdec_contract.py",
        "tests/test_data.py",
        "tests/test_mvdec_architecture.py",
        "tests/test_mvdec_trajectory.py",
    ],
    cwd=REPO,
    env={**os.environ, "PYTHONPATH": os.pathsep.join([str(REPO), str(REPO / "src")])},
    check=True,
)

In [ ]:
for protocol, dataset, seed in RUNS:
    command = [
        sys.executable, "-u", "src/representation_learning/MVDEC_dense.py", dataset,
        "--runs", "1", "--seed", str(seed),
        "--protocol", protocol,
        "--progress-interval", "100",
        "--dataset-root", "external_repos/DEKM/datasets",
        "--public-output-dir", str(OUTPUT),
    ]
    print("Running:", " ".join(command), flush=True)
    tag = f"{protocol.split('_true_bottleneck_')[1]}_{dataset}_seed_{seed}"
    trajectory = RESULTS / "trajectory" / f"{tag}.jsonl"
    env = {**os.environ, "MVDEC_LABEL_TRAJECTORY_LOG": str(trajectory)}
    with (RESULTS / "logs" / f"{tag}.log").open("w", encoding="utf-8") as log:
        result = subprocess.run(command, cwd=REPO, env=env, check=False, timeout=4 * 60 * 60, stdout=log, stderr=subprocess.STDOUT)
    print(tag, "exit code", result.returncode, flush=True)

In [ ]:
import statistics

PAPER = {
    "REUTERS": {"view1": (0.7642, 0.5999), "view2": (0.6501, 0.5129), "fused": (0.7813, 0.6141)},
    "20NEWS": {"view1": (0.4120, 0.4376), "view2": (0.4086, 0.4032), "fused": (0.4163, 0.4543)},
}
rows = []
for path in sorted(OUTPUT.rglob("manifest.json")):
    manifest = json.loads(path.read_text(encoding="utf-8"))
    if manifest.get("protocol_id") not in PROTOCOLS or manifest.get("status") != "complete":
        continue
    rows.append((manifest["protocol_id"].split("_true_bottleneck_")[1] + " " + manifest["dataset"], manifest["seed"], manifest["metrics"]))
    # Keep only small text records; weights and latents stay in /kaggle/temp.
    target = RESULTS / "runs" / path.parent.relative_to(OUTPUT)
    target.mkdir(parents=True, exist_ok=True)
    for small in path.parent.iterdir():
        if small.is_file() and small.suffix in {".json", ".csv", ".txt"}:
            shutil.copy2(small, target / small.name)
rows.sort(key=lambda row: (row[0], row[1]))
print("complete runs:", len(rows), "of", len(RUNS))
for dataset in sorted({row[0] for row in rows}):
    picked = [metrics for d, _, metrics in rows if d == dataset]
    print()
    print(dataset)
    for d, seed, metrics in rows:
        if d == dataset:
            print(seed, "  ".join("%s %.4f/%.4f" % (k, metrics[k + "_acc"], metrics[k + "_nmi"]) for k in PAPER[dataset.split()[-1]]))
    print("rep    mean_ACC best_ACC | paper_ACC | mean_NMI best_NMI | paper_NMI")
    for key, (paper_acc, paper_nmi) in PAPER[dataset.split()[-1]].items():
        accs = [m[key + "_acc"] for m in picked]
        nmis = [m[key + "_nmi"] for m in picked]
        print("%-6s %8.4f %8.4f | %.4f | %8.4f %8.4f | %.4f" % (key, statistics.mean(accs), max(accs), paper_acc, statistics.mean(nmis), max(nmis), paper_nmi))

zip_path = Path(shutil.make_archive(str(RESULTS) + "-results", "zip", RESULTS.parent, RESULTS.name))
digest = file_sha256(zip_path)
Path(str(zip_path) + ".sha256").write_text(digest + "  " + zip_path.name + chr(10), encoding="ascii")
print("ZIP:", zip_path)
print("SHA-256:", digest)